# Classwork: The Analyst's Toolkit

**Names:** _(pair)_

After learning about different tools (or a toolkit-- Descriptive statistics, Correlation, Regression/Multiple regression, Chi-square test) of working with data in the previous Mods, in this acitivity you will apply your real skills on the data you have.  
  
  Suppose your company hands you `customers.csv` (40 customers: `age`, `income`, `spending`, `region`, `subscribed`) and a list of questions. You are the analyst. Answer each one with the right tool/technique you decide to apply.

Work in pairs. Write answers in the markdown cells provided.

## Setup
Import the libraries you'll need and load the data.

In [1]:
# import pandas (and numpy, scipy.stats, statsmodels.api as you need them)
# load customers.csv into a DataFrame
# take a first look (head, info)
import pandas as pd
import scipy.stats as st
import numpy as np

df = pd.read_csv("./customers.csv")

df.head(5)
print()
df.info()


<class 'pandas.DataFrame'>
RangeIndex: 40 entries, 0 to 39
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   customer_id  40 non-null     int64  
 1   age          40 non-null     int64  
 2   income       38 non-null     float64
 3   spending     40 non-null     float64
 4   region       40 non-null     str    
 5   subscribed   40 non-null     str    
dtypes: float64(2), int64(2), str(2)
memory usage: 2.0 KB


## Part 0: Know your data, then clean it

This data is dirty on purpose. Find and fix **three** problems before you analyze anything.

In [2]:
# HINT 1: a numeric column has missing values -> decide how to fill them (mean? median? why?)

# tip: check .isna().sum() and the unique values of the category column


print(df.isna().sum())
#this told me there are 2 blank spots on 'income'
print(df['income'].describe())
#Using this to see my mean and median




customer_id    0
age            0
income         2
spending       0
region         0
subscribed     0
dtype: int64
count     38.000000
mean      83.007895
std       27.628563
min      -50.000000
25%       72.975000
50%       84.700000
75%       98.500000
max      114.500000
Name: income, dtype: float64


In [3]:
# HINT 2: one category label is written in the wrong case -> standardize it

print(df['region'].value_counts())
#this shwoed me a single 'east' when the rest are 'East'

#map for refactoring east
region_map = {
    "east":"East", "est":"East"
}

df['region'] = df['region'].replace(region_map)


region
East     20
West     10
South     9
east      1
Name: count, dtype: int64


In [4]:
# HINT 3: one value is impossible for its column -> treat it as missing, then fill

print(df['income'].describe()) #shows min of -50
df.loc[df['income'] < 0, 'income'] = np.nan

df['income'] = df['income'].fillna(df['income'].median())
#I chose to fill in the blanks with the mean because my mean and median are closed together, which means there are not a lot of outliers skweing the data

count     38.000000
mean      83.007895
std       27.628563
min      -50.000000
25%       72.975000
50%       84.700000
75%       98.500000
max      114.500000
Name: income, dtype: float64


**Explain each issue:** what you found and how you handled it.

issue 1: 2 missing values in 'income.' Fixed by replacing the missing vals with the mean, this choice was made becasue mean and median of col were close together

issue 2: 'region' column had 'east' and 'East'. fixed by creating a map used in a .replace function to turn the lowercase east into uppercase East. decision was made because the other asnwers were capital West and South

Issue 3: Impossible value of -50 on the 'income' column. found it, turned into a nan, then re-ran the code from issue one to turn it into the mean. 


## Part 1: Pick the tool for each question

For each question: (a) name the data types involved, (b) name the tool, (c) run it, (d) answer in one plain sentence.

### Q1. Do customers who earn more also spend more?

In [5]:
# data types: income = ?, spending = ?
# tool: ?
# hint: two continuous variables moving together

#income is numerical and so is spending
#tool: Correlaion
R, P = st.pearsonr(df['income'], df['spending'])
print(f"R = {R:.3f}, P = {P:.3f}")



R = 0.717, P = 0.000


*Your answer:* With a P value of under 0.05, and an R of 0.7, we can see that the two variables have a positive relation NOT due to chance. This tells us that customer that earn more are 'more likely' to spend more 


### Q2. Is a customer's region related to whether they subscribed?

In [6]:
# data types: region = ?, subscribed = ?
# tool: ?
# hint: two categorical variables -> build a table with pd.crosstab, then test it

#both columns are categorical
#tool: chi square? 

table = pd.crosstab(df['region'], df['subscribed'])
print(table)
chi2, p, dof, expected = st.chi2_contingency(table)
print()
print(f"{p:.3f}")


subscribed  No  Yes
region             
East         5   16
South        6    3
West         9    1

0.001


*Your answer:* With a p value of under 0.05, we can see that the two variables are related


### Q3. Can we predict spending from income and age together? How much do they explain? (You can review your Assignment 4 for this)

In [7]:
import statsmodels.api as sm

In [8]:
# data types: predicting a continuous outcome from TWO predictors
# tool: ?
# hint: remember to add a constant; look at R-squared and each predictor's p-value

#tool: multiple regression -> we're looking for a prediction from 2 vars

X = df[['income', 'age']]
X = sm.add_constant(X)
y = df['spending']

model = sm.OLS(y, X).fit()
print(model.summary())


                            OLS Regression Results                            
Dep. Variable:               spending   R-squared:                       0.520
Model:                            OLS   Adj. R-squared:                  0.494
Method:                 Least Squares   F-statistic:                     20.04
Date:                Mon, 05 Oct 2026   Prob (F-statistic):           1.27e-06
Time:                        16:12:38   Log-Likelihood:                -131.11
No. Observations:                  40   AIC:                             268.2
Df Residuals:                      37   BIC:                             273.3
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const         19.2622      6.000      3.210      0.0

*Your answer:* income and age together explain about 52% (R-squared) of the variation in spending, but income is the one doing the work (p-value: 0.003) while age isn't really contributing (p-value: 0.515). Overall, we can use income to somewhat predict spending, but age isn't relaly helpful


### Q4. Which region has the most customers, and what is the average income overall?

In [13]:
# hint: count a categorical column
# take the mean of a numeric column

#most customers
print(df['region'].value_counts())

#AVG income
print(df['income'].mean())


region
East     21
West     10
South     9
Name: count, dtype: int64
86.46


*Your answer:* East is the region with the most customers. AVG income is 86.46


## Part 2: The reveal

Answer these two in a sentence each (no code needed).

**a)** Look back at your four tools (Correlation, Chi-square test, Multiple regression, & Descriptive statistic)). What decided which tool you used each time? (Hint: it was not the wording of the question.)

*Your answer:* The Data Type of each column was a big indicator, because you can not run correlation. It also depended on what the question wanted, just a relation? or a prediction? 2 different things. 


**b)** In Q1 you found a relationship. Can you tell the company that higher income *causes* higher spending? Why or why not?

*Your answer:* No. All that we got from Q1 is that there is a relation, this does not prove Causation. From that test we can only show that as one moves so does the other. 


## Bonus

In Q2 you ran a chi-square test. Look at its **expected counts** (not the observed counts). Do they all meet the assumption you learned in the chi-square module? What would you tell the company about how much to trust that result?

In [10]:
# hint: chi2_contingency returns the expected table as its 4th value
# check: are all expected counts >= 5?


*Your answer:*


### Push your code to GitHub